# Deep Learning with Keras and TensorFlow — Course Summary

**IBM AI Engineering Professional Certificate — Deep Learning with Keras and TensorFlow**

This notebook provides a **summary** of all lab materials from the *Deep Learning with Keras and TensorFlow* course (Functional API, custom layers, transfer learning, transformers, generative models, custom training, hyperparameter tuning, and reinforcement learning).

---
## Summary Table: Topics and Key Notebooks
---

| Topic | Notebooks | Main idea |
|-------|-----------|-----------|
| Functional API | m01l01_lab_implementing_the_functional_api_in_keras | Build models with Keras Functional API (input, hidden, output) |
| Custom layers & models | m01l02_lab_creating_custom_layers_and_models | Subclass Layer/Model for custom logic |
| Transfer learning | m02l02_lab_transfer_learning_implementation | Use pre-trained models, freeze/fine-tune |
| Transpose convolution | lab_practical_application_of_transpose_convolution | Upsampling, Conv2DTranspose for segmentation/decoders |
| Advanced transformers | review-lab_building_advanced_transformers | Build and use transformer architectures |
| Text generation | m03l02_lab_implementing_transformers_for_text_genera | Transformers for text generation |
| Autoencoders | m04l01_lab_building_autoencoders | Encoder–decoder, reconstruction loss |
| Diffusion | m04_lab_implementing_diffusion_models | Diffusion models for generation |
| GANs | m04l02_lab_develop_gans_using_keras | Generator and discriminator with Keras |
| Custom training | m05l01_lab_custom_training_loops_in_keras | Custom train_step / training loop |
| Hyperparameter tuning | m05l02_lab_hyperparameter_tuning_with_keras_tuner | Keras Tuner for hyperparameter search |
| Q-Learning & DQN | m06l01_lab_implementing_q-learning_in_keras, m06l01_lab_building_a_deep_q-network_with_keras | Q-Learning and Deep Q-Network in Keras |
| Practice project | practice_project_fruit_classification_using_tf | Fruit image classification with TF |
| Final project | final_proj-classify_waste_products_using_tl_ft_(1) | Classify waste products using transfer learning / fine-tuning |

## Lab-by-Lab Summary (Objective, Strategy, Consolidated Code)
---

For each lab, the following format is used:
- **Objective:** Taken from the lab's Learning objectives section.
- **Implementation Strategy:** How the code achieves the objective (model, data, training).
- **Consolidated Code Block:** Single runnable cell with imports, model, and training (no extra prints/plots).

<a id='m01l01_lab_implementing_the_functional_api_in_keras'></a>

### Lab: Implementing the Functional API in Keras (m01l01_lab_implementing_the_functional_api_in_keras)

**Objective:** Use the Keras Functional API to build a neural network; create input, hidden, and output layers.

**Implementation Strategy:**
- Define an input layer with `keras.Input()`; chain Dense (and other) layers by calling them on previous outputs.
- Build a `Model` with inputs and outputs; compile and fit as usual.

In [2]:
# Consolidated: Functional API (m01l01_lab_implementing_the_functional_api_in_keras)
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

input_layer = Input(shape=(20,))
hidden1 = Dense(64, activation='relu')(input_layer)
hidden2 = Dense(64, activation='relu')(hidden1)
output_layer = Dense(1, activation='sigmoid')(hidden2)
model = Model(inputs=input_layer, outputs=output_layer)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
X = np.random.rand(1000, 20).astype(np.float32)
y = np.random.randint(2, size=(1000, 1)).astype(np.float32)
model.fit(X, y, epochs=3, batch_size=32)

Epoch 1/3
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5060 - loss: 0.7016   
Epoch 2/3
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 769us/step - accuracy: 0.5390 - loss: 0.6903
Epoch 3/3
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 762us/step - accuracy: 0.5400 - loss: 0.6860


### Lab: Creating Custom Layers and Models (m01l02_lab_creating_custom_layers_and_models)

**Objective:** Create custom layers and models by subclassing Keras Layer and Model.

**Implementation Strategy:**
- Subclass `tf.keras.layers.Layer`; implement `__init__` and `call`; optionally `build` for weight creation.
- Subclass `tf.keras.Model` to define full models with custom forward logic and training behavior.

In [3]:
# Consolidated: Custom layers and models (m01l02_lab_creating_custom_layers_and_models)
import tensorflow as tf

class MyDenseLayer(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
    def build(self, input_shape):
        self.w = self.add_weight(shape=(input_shape[-1], self.units), initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(shape=(self.units,), initializer='zeros', trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        return tf.nn.relu(tf.matmul(inputs, self.w) + self.b)

model = tf.keras.Sequential([MyDenseLayer(32), MyDenseLayer(10)])
model.build((None, 20))
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
X = tf.random.normal((100, 20))
y = tf.random.uniform((100,), 0, 10, dtype=tf.int32)
model.fit(X, y, epochs=2, batch_size=16)

Epoch 1/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0900 - loss: 10.6619      
Epoch 2/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1100 - loss: 10.1129


### Lab: Transfer Learning Implementation (m02l02_lab_transfer_learning_implementation)

**Objective:** Implement transfer learning using a pre-trained model; freeze base and train a new head.

**Implementation Strategy:**
- Load a pre-trained model (e.g. from Keras Applications); set base layers to non-trainable.
- Add new classification layers on top; compile and train on the target dataset; optionally unfreeze and fine-tune.

In [4]:
# Consolidated: Transfer learning (m02l02_lab_transfer_learning_implementation)
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import numpy as np

base = MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights='imagenet')
base.trainable = False
x = base.output
x = GlobalAveragePooling2D()(x)
out = Dense(5, activation='softmax')(x)
model = Model(inputs=base.input, outputs=out)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
X = np.random.rand(80, 96, 96, 3).astype(np.float32)
y = np.random.randint(0, 5, size=(80,))
model.fit(X, y, epochs=2, batch_size=16)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/2
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.1375 - loss: 1.9487
Epoch 2/2
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1750 - loss: 1.8006


### Lab: Practical Application of Transpose Convolution (lab_practical_application_of_transpose_convolution)

**Objective:** Apply transpose convolution (Conv2DTranspose) for upsampling in practical tasks (e.g. segmentation or decoders).

**Implementation Strategy:**
- Use `Conv2DTranspose` (or equivalent) to increase spatial dimensions; combine with normal convolutions in decoder/U-Net-style architectures.
- Train on a task that requires spatial upsampling (e.g. image reconstruction or segmentation).

In [5]:
# Consolidated: Transpose convolution (lab_practical_application_of_transpose_convolution)
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, Input
from tensorflow.keras.models import Model
import numpy as np

inp = Input(shape=(8, 8, 32))
x = Conv2DTranspose(16, 3, strides=2, padding='same', activation='relu')(inp)
out = Conv2D(1, 3, padding='same', activation='sigmoid')(x)
model = Model(inp, out)
model.compile(optimizer='adam', loss='mse')
X = np.random.rand(32, 8, 8, 32).astype(np.float32)
y = np.random.rand(32, 16, 16, 1).astype(np.float32)
model.fit(X, y, epochs=2, batch_size=8)

Epoch 1/2
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0877  
Epoch 2/2
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0863


### Lab: Building Advanced Transformers (review-lab_building_advanced_transformers)

**Objective:** Build advanced transformer-based models (attention, multi-head attention, encoder/decoder blocks).

**Implementation Strategy:**
- Implement or use Keras transformer building blocks (attention, feed-forward, layer norm); stack encoder/decoder layers.
- Apply to a sequence task (e.g. classification or sequence-to-sequence).

In [6]:
# Consolidated: Building advanced transformers (review-lab_building_advanced_transformers)
import tensorflow as tf
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, Dense, Dropout, Input
from tensorflow.keras.models import Model
import numpy as np

def transformer_block(x, d_model=64, num_heads=4, ff_dim=128):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)(x, x)
    x = LayerNormalization()(x + Dropout(0.1)(attn))
    ffn = Dense(ff_dim, activation='relu')(x)
    ffn = Dense(d_model)(ffn)
    return LayerNormalization()(x + Dropout(0.1)(ffn))

inp = Input(shape=(10, 64))
x = transformer_block(inp)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
out = Dense(3, activation='softmax')(x)
model = Model(inp, out)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
X = np.random.rand(50, 10, 64).astype(np.float32)
y = np.random.randint(0, 3, size=(50,))
model.fit(X, y, epochs=2, batch_size=8)

Epoch 1/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2800 - loss: 1.2197
Epoch 2/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3400 - loss: 1.1993


### Lab: Implementing Transformers for Text Generation (m03l02_lab_implementing_transformers_for_text_genera)

**Objective:** Use transformer models for text generation.

**Implementation Strategy:**
- Build or use a transformer decoder (e.g. causal self-attention); train on text data with next-token prediction.
- Generate text by autoregressive sampling from the model.

In [7]:
# Consolidated: Transformers for text generation (m03l02_lab_implementing_transformers_for_text_genera)
import tensorflow as tf
from tensorflow.keras.layers import Embedding, MultiHeadAttention, Dense, LayerNormalization, Input
from tensorflow.keras.models import Model
import numpy as np

vocab_size, seq_len, d_model = 1000, 20, 64
inp = Input(shape=(seq_len,))
x = Embedding(vocab_size, d_model)(inp)
attn = MultiHeadAttention(num_heads=4, key_dim=d_model // 4)(x, x)
x = LayerNormalization()(x + attn)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
out = Dense(vocab_size, activation='softmax')(x)
model = Model(inp, out)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
X = np.random.randint(0, vocab_size, size=(64, seq_len))
y = np.random.randint(0, vocab_size, size=(64,))
model.fit(X, y, epochs=2, batch_size=8)

Epoch 1/2
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0000e+00 - loss: 6.9121  
Epoch 2/2
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3438 - loss: 6.6352


### Lab: Building Autoencoders (m04l01_lab_building_autoencoders)

**Objective:** Build autoencoders (encoder–decoder) for reconstruction or representation learning.

**Implementation Strategy:**
- Define encoder (downsample to latent) and decoder (upsample to input size); train with reconstruction loss (e.g. MSE or BCE).
- Optionally use for denoising, dimensionality reduction, or as a prior for generative models.

In [8]:
# Consolidated: Autoencoders (m04l01_lab_building_autoencoders)
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Model
import numpy as np

encoding_dim = 16
inp = Input(shape=(784,))
encoded = Dense(encoding_dim, activation='relu')(inp)
decoded = Dense(784, activation='sigmoid')(encoded)
autoencoder = Model(inp, decoded)
autoencoder.compile(optimizer='adam', loss='mse')
X = np.random.rand(200, 784).astype(np.float32)
autoencoder.fit(X, X, epochs=2, batch_size=32)

Epoch 1/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0839  
Epoch 2/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0832


### Lab: Implementing Diffusion Models (m04_lab_implementing_diffusion_models)

**Objective:** Implement a diffusion model for generation (e.g. images).

**Implementation Strategy:**
- Implement forward (noising) and reverse (denoising) process; train a network to predict noise or x0.
- Sample by iteratively denoising from random noise using the trained model.

In [9]:
# Consolidated: Diffusion models — minimal UNet-style denoiser (m04_lab_implementing_diffusion_models)
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Input, Dense, Flatten, Reshape
from tensorflow.keras.models import Model
import numpy as np

inp = Input(shape=(14, 14, 64))
x = Conv2D(32, 3, padding='same', activation='relu')(inp)
x = Conv2D(64, 3, padding='same', activation='relu')(x)
out = Conv2D(64, 3, padding='same')(x)
model = Model(inp, out)
model.compile(optimizer='adam', loss='mse')
X = np.random.rand(32, 14, 14, 64).astype(np.float32)
y = np.random.rand(32, 14, 14, 64).astype(np.float32)
model.fit(X, y, epochs=2, batch_size=8)

Epoch 1/2
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2635  
Epoch 2/2
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1307 


### Lab: Develop GANs Using Keras (m04l02_lab_develop_gans_using_keras)

**Objective:** Develop Generative Adversarial Networks (GANs) with Keras.

**Implementation Strategy:**
- Build a generator (e.g. dense/conv from latent to image) and a discriminator (image to real/fake).
- Train with alternating or combined steps: discriminator on real vs fake; generator to fool discriminator.

In [10]:
# Consolidated: GANs — generator + discriminator (m04l02_lab_develop_gans_using_keras)
import tensorflow as tf
from tensorflow.keras.layers import Dense, Reshape, Conv2D, Flatten, LeakyReLU, Input
from tensorflow.keras.models import Model
import numpy as np

latent_dim = 64
# Generator
z = Input(shape=(latent_dim,))
g = Dense(7*7*64, activation='relu')(z)
g = Reshape((7, 7, 64))(g)
g = Conv2D(32, 3, padding='same', activation='relu')(g)
g_out = Conv2D(1, 3, padding='same', activation='sigmoid')(g)
generator = Model(z, g_out)
# Discriminator
img = Input(shape=(7, 7, 1))
d = Flatten()(img)
d = Dense(64, activation=LeakyReLU(0.2))(d)
d_out = Dense(1, activation='sigmoid')(d)
discriminator = Model(img, d_out)
discriminator.compile(optimizer='adam', loss='binary_crossentropy')
discriminator.trainable = False
gan = Model(z, discriminator(generator(z)))
gan.compile(optimizer='adam', loss='binary_crossentropy')
z_batch = np.random.randn(16, latent_dim).astype(np.float32)
fake = generator(z_batch)
discriminator.train_on_batch(fake, np.zeros((16, 1)))

/Users/umutkucuk/Projects/personal_projects/Courses/.venv/lib/python3.12/site-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


array(0.46278548, dtype=float32)

### Lab: Custom Training Loops in Keras (m05l01_lab_custom_training_loops_in_keras)

**Objective:** Implement custom training loops (e.g. overriding train_step) in Keras.

**Implementation Strategy:**
- Subclass `Model` and override `train_step` (and optionally `test_step`) to define custom loss and updates.
- Use `GradientTape` for gradients and apply optimizer updates manually if needed.

In [11]:
# Consolidated: Custom training loop (m05l01_lab_custom_training_loops_in_keras)
import tensorflow as tf
import numpy as np

class CustomModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(32, activation='relu')
        self.dense2 = tf.keras.layers.Dense(10, activation='softmax')
    def call(self, x):
        return self.dense2(self.dense1(x))
    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            loss = tf.keras.losses.sparse_categorical_crossentropy(y, y_pred)
            loss = tf.reduce_mean(loss)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return {'loss': loss}

model = CustomModel()
model.compile(optimizer='adam')
X = tf.random.normal((64, 20))
y = tf.random.uniform((64,), 0, 10, dtype=tf.int32)
model.fit(tf.data.Dataset.from_tensor_slices((X, y)).batch(16), epochs=2)

Epoch 1/2
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 927us/step - loss: 0.0000e+00
Epoch 2/2
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 854us/step - loss: 0.0000e+00


### Lab: Hyperparameter Tuning with Keras Tuner (m05l02_lab_hyperparameter_tuning_with_keras_tuner)

**Objective:** Set up Keras Tuner and run hyperparameter search (e.g. units, learning rate).

**Implementation Strategy:**
- Define a model-building function that takes `hp` and returns a compiled model; choose a tuner (RandomSearch, Hyperband, etc.).
- Run `tuner.search()` on data; retrieve best hyperparameters and retrain or use best model.

In [13]:
# Consolidated: Keras Tuner (m05l02_lab_hyperparameter_tuning_with_keras_tuner)
import tensorflow as tf
import keras_tuner as kt
import numpy as np

def build_model(hp):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(units=hp.Int('units', 32, 128, step=32), activation='relu', input_shape=(20,)),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

tuner = kt.Hyperband(build_model, objective='val_accuracy', max_epochs=2, factor=2)
X_train = np.random.rand(200, 20).astype(np.float32)
y_train = np.random.randint(0, 10, size=(200,))
X_val = np.random.rand(50, 20).astype(np.float32)
y_val = np.random.randint(0, 10, size=(50,))
tuner.search(X_train, y_train, validation_data=(X_val, y_val), epochs=2)
best_hps = tuner.get_best_hyperparameters(1)[0]

Trial 5 Complete [00h 00m 01s]
val_accuracy: 0.11999999731779099

Best val_accuracy So Far: 0.11999999731779099
Total elapsed time: 00h 00m 03s


### Lab: Implementing Q-Learning in Keras (m06l01_lab_implementing_q-learning_in_keras)

**Objective:** Implement Q-Learning (tabular or function approximation) using Keras.

**Implementation Strategy:**
- Define an environment (or use a gym-like one); build a Q-network or table; implement epsilon-greedy policy and Q-updates.
- Train the agent by interacting with the environment and updating Q-values (e.g. target = reward + gamma * max_a Q(s',a)).

In [14]:
# Consolidated: Q-Learning in Keras (m06l01_lab_implementing_q-learning_in_keras)
import tensorflow as tf
import numpy as np

n_states, n_actions = 16, 4
model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(n_states,)),
    tf.keras.layers.Dense(n_actions)
])
model.compile(optimizer='adam', loss='mse')
s = np.eye(n_states)[np.random.randint(0, n_states)]
s = s.reshape(1, -1).astype(np.float32)
q = model(s)
target = np.random.rand(1, n_actions).astype(np.float32)
model.train_on_batch(s, target)

array(0.67823416, dtype=float32)

### Lab: Building a Deep Q-Network with Keras (m06l01_lab_building_a_deep_q-network_with_keras)

**Objective:** Implement a Deep Q-Network (DQN) with Keras to solve an RL problem.

**Implementation Strategy:**
- Use a neural network to approximate Q(s,a); implement experience replay and (optionally) a target network.
- Train with mini-batches from replay buffer; minimize TD error (e.g. MSE between Q(s,a) and target).

In [15]:
# Consolidated: Deep Q-Network (m06l01_lab_building_a_deep_q-network_with_keras)
import tensorflow as tf
import numpy as np

class DQN(tf.keras.Model):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.d1 = tf.keras.layers.Dense(64, activation='relu')
        self.d2 = tf.keras.layers.Dense(64, activation='relu')
        self.d3 = tf.keras.layers.Dense(action_dim)
    def call(self, x):
        return self.d3(self.d2(self.d1(x)))

state_dim, action_dim = 8, 4
model = DQN(state_dim, action_dim)
model.build((None, state_dim))
optimizer = tf.keras.optimizers.Adam(0.001)
states = tf.random.normal((32, state_dim))
with tf.GradientTape() as tape:
    q = model(states)
    targets = tf.random.normal((32, action_dim))
    loss = tf.reduce_mean(tf.square(q - targets))
grads = tape.gradient(loss, model.trainable_variables)
optimizer.apply_gradients(zip(grads, model.trainable_variables))

/Users/umutkucuk/Projects/personal_projects/Courses/.venv/lib/python3.12/site-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'dqn', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


<Variable path=adam/iteration, shape=(), dtype=int64, value=1>

### Practice Project: Fruit Classification Using TF (practice_project_fruit_classification_using_tf)

**Objective:** Set up and organize a fruit image dataset; build a classifier using TensorFlow.

**Implementation Strategy:**
- Load and preprocess fruit images (e.g. resize, augmentation); build a CNN (or use transfer learning).
- Train and evaluate; report accuracy and optionally visualize results.

In [16]:
# Consolidated: Fruit classification (practice_project_fruit_classification_using_tf)
import tensorflow as tf
import numpy as np

model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, 3, activation='relu', input_shape=(64, 64, 3)),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(5, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
X = np.random.rand(100, 64, 64, 3).astype(np.float32)
y = np.random.randint(0, 5, size=(100,))
model.fit(X, y, epochs=2, batch_size=16)

Epoch 1/2


/Users/umutkucuk/Projects/personal_projects/Courses/.venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1300 - loss: 3.6131  
Epoch 2/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2100 - loss: 1.7685 


### Final Project: Classify Waste Products Using TL/FT (final_proj-classify_waste_products_using_tl_ft_(1))

**Objective:** Classify waste products using transfer learning and/or fine-tuning.

**Implementation Strategy:**
- Use a pre-trained image model; add a new head for waste categories; optionally fine-tune part of the base.
- Train on waste image data; evaluate classification performance.

In [17]:
# Consolidated: Waste classification with TL (final_proj-classify_waste_products_using_tl_ft_(1))
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import numpy as np

base = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
base.trainable = False
x = base.output
x = GlobalAveragePooling2D()(x)
out = Dense(4, activation='softmax')(x)
model = Model(inputs=base.input, outputs=out)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
X = np.random.rand(64, 128, 128, 3).astype(np.float32)
y = np.random.randint(0, 4, size=(64,))
model.fit(X, y, epochs=2, batch_size=8)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/2
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.1875 - loss: 1.5476
Epoch 2/2
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3125 - loss: 1.4627
